# Module 9: AI System Design

# Chapter 2: Enterprise RAG System Design

So far we have learned:

```text
System Design Fundamentals

↓

Scalability

↓

Reliability

↓

Caching

↓

Microservices

↓

Observability
```

Now we combine these concepts to design one of the most common AI systems asked in interviews:

> **Design a Production Enterprise RAG System**

This is one of the highest-frequency design questions for AI Engineers, Applied Scientists, GenAI Engineers, and ML Engineers.

---

# What is Enterprise RAG?

Enterprise RAG is a system that allows an LLM to answer questions using an organization's private knowledge.

Instead of relying only on its pretrained parameters,

the LLM retrieves:

```text
Company Documents

Policies

Contracts

Research Papers

Internal Wikis

Emails

Knowledge Bases
```

before generating an answer.

---

## Interview Takeaway

Enterprise RAG augments an LLM with private organizational knowledge.

---

# High-Level Architecture

```text
                Users
                   │
                   ▼
             API Gateway
                   │
                   ▼
          Authentication
                   │
                   ▼
             RAG Service
                   │
      ┌────────────┼─────────────┐
      ▼            ▼             ▼
 Retriever     Conversation    Cache
                 Memory
      │
      ▼
 Hybrid Search
      │
 ┌────┴─────────────┐
 ▼                  ▼
Vector DB     Keyword Search
      │
      └──────┬──────┘
             ▼
        Reranker
             ▼
       Prompt Builder
             ▼
            LLM
             ▼
      Guardrails
             ▼
         Response
```

---

# Why Not Fine-Tune?

Question:

```text
Why not simply fine-tune the model?
```

Problems

```text
Expensive

Slow

Knowledge Changes Frequently

Hard To Maintain

Model Retraining Required
```

RAG instead:

```text
Update Documents

↓

No Retraining
```

---

## Interview Takeaway

RAG separates knowledge from model parameters.

---

# Enterprise Requirements

Typical requirements:

Functional

```text
Document Upload

Semantic Search

Question Answering

Source Citations

Conversation History
```

---

Non-Functional

```text
< 2 Second Latency

99.9% Availability

Secure Access

Scalable

Low Cost
```

---

# Overall Pipeline

```text
Upload Documents

↓

Ingestion Pipeline

↓

Chunking

↓

Embedding

↓

Vector Database

↓

Retriever

↓

Reranker

↓

Prompt Builder

↓

LLM

↓

Response
```

Every stage can become an interview topic.

---

# Stage 1 — Document Ingestion

Documents may arrive from:

```text
PDF

Word

PowerPoint

Excel

Confluence

SharePoint

Web Pages

Emails

Databases
```

Need connectors.

---

Architecture

```text
Source

↓

Connector

↓

Raw Storage
```

---

## Interview Takeaway

Ingestion is responsible for collecting enterprise knowledge.

---

# Document Parsing

Raw files cannot be embedded directly.

Need parsing.

Extract:

```text
Text

Tables

Metadata

Images

Sections
```

---

Example

PDF

↓

```text
Plain Text

Title

Page Number

Author
```

---

Metadata becomes useful later.

---

# OCR Pipeline

Scanned documents require OCR.

Pipeline

```text
Image

↓

OCR

↓

Text

↓

Chunking
```

Examples

```text
Tesseract

Azure Document Intelligence

Google Document AI
```

---

## Interview Takeaway

OCR converts scanned documents into searchable text.

---

# Cleaning Pipeline

Remove

```text
Headers

Footers

Duplicate Text

Watermarks

Noise
```

Normalize

```text
Whitespace

Encoding

Formatting
```

---

Poor cleaning

↓

Poor retrieval.

---

# Metadata Extraction

Store metadata such as:

```text
Document ID

Department

Author

Security Level

Creation Date

Tags

Language
```

---

Example

```json
{
"title":"Leave Policy",

"department":"HR",

"security":"Internal",

"version":"2.1"
}
```

---

## Interview Takeaway

Metadata enables filtering and access control.

---

# Chunking

Never embed an entire document.

Instead split into chunks.

Example

```text
200 Page PDF

↓

800 Chunks
```

---

Common chunk sizes

```text
256 Tokens

512 Tokens

1024 Tokens
```

---

Trade-off

Small chunks

↓

Better precision

Large chunks

↓

Better context

---

## Interview Takeaway

Chunk size significantly impacts retrieval quality.

---

# Chunking Strategies

Fixed Size

```text
Every 512 Tokens
```

---

Recursive

Split by

```text
Heading

↓

Paragraph

↓

Sentence
```

---

Semantic Chunking

Split where topic changes.

---

Parent-Child Chunking

```text
Small Child Chunk

↓

Linked To

↓

Large Parent Chunk
```

---

## Interview Takeaway

Semantic and recursive chunking usually outperform fixed-size chunking.

---

# Embedding Pipeline

Every chunk becomes

```text
Vector
```

Pipeline

```text
Chunk

↓

Embedding Model

↓

Dense Vector

↓

Vector Database
```

---

Popular Models

```text
OpenAI

BGE

E5

GTE

Jina Embeddings

Cohere
```

---

# Why Embeddings?

LLMs understand language.

Vector databases understand numbers.

Embedding bridges the gap.

```text
Text

↓

Embedding

↓

1536-D Vector
```

---

# Embedding Storage

Each chunk stores

```text
Embedding

Text

Metadata

Document ID
```

Example

```json
{

vector:[...]

text:"Vacation policy"

department:"HR"

page:12
}
```

---

# Vector Database

Purpose

```text
Store

Search

Embeddings
```

Examples

```text
Pinecone

Milvus

Weaviate

Qdrant

Azure AI Search

Databricks Vector Search

FAISS
```

---

## Interview Takeaway

Vector databases enable semantic retrieval.

---

# Indexing

After embeddings,

build ANN index.

Examples

```text
HNSW

IVF

PQ

DiskANN
```

Without indexes,

search becomes slow.

---

## Interview Takeaway

ANN indexes make vector search scalable.

---

# Query Pipeline

User asks

```text
"What is maternity leave?"
```

Pipeline

```text
Query

↓

Embedding

↓

Vector Search

↓

Top K Results
```

---

# Hybrid Search

Production systems rarely use only vectors.

Instead combine

```text
Semantic Search

+

Keyword Search
```

Architecture

```text
User Query

↓

Vector Search

↓

BM25

↓

Merge Results
```

---

Advantages

```text
Higher Recall

Better Precision
```

---

## Interview Takeaway

Hybrid search is the enterprise standard.

---

# Metadata Filtering

Suppose

```text
HR Employee
```

Should not access

```text
Finance Documents
```

Apply filters

```text
Department

Security Level

Language

Country
```

before retrieval.

---

## Interview Takeaway

Metadata filtering enforces access control during retrieval.

---

# Top-K Retrieval

Retriever returns

```text
Top K Chunks
```

Example

```text
Top 10
```

Question

Should all 10 be sent?

Usually

```text
No
```

Need reranking.

---

# Reranking

Retriever optimizes

```text
Recall
```

Reranker optimizes

```text
Precision
```

Pipeline

```text
Top 20

↓

Cross Encoder

↓

Top 5
```

---

Popular Models

```text
BGE Reranker

Cohere Rerank

Jina Reranker

Cross Encoders
```

---

## Interview Takeaway

Rerankers improve answer quality by selecting the most relevant documents.

---

# Prompt Builder

Construct final prompt.

Example

```text
System Prompt

+

Retrieved Context

+

Conversation History

+

User Question
```

↓

LLM

---

Prompt quality greatly impacts responses.

---

# Context Window

Cannot send

```text
Entire Database
```

Need

```text
Context Selection
```

Only highest-quality chunks fit into:

```text
Model Context Window
```

---

# Conversation Memory

Store

```text
Previous Questions

Preferences

Conversation Summary
```

Need memory

↓

Better multi-turn conversations.

---

# Response Generation

LLM receives

```text
Question

+

Retrieved Context

+

Instructions
```

Returns

```text
Grounded Answer
```

---

# Citation Generation

Always return sources.

Example

```text
Answer

↓

Page 12

↓

HR Policy

↓

Version 3.2
```

Benefits

```text
Trust

Auditability
```

---

# Caching

Cache

```text
Embeddings

Retrieved Documents

LLM Responses
```

Benefits

```text
Lower Cost

Lower Latency
```

---

# Security

Need

```text
Authentication

Authorization

Document Permissions

Prompt Guardrails

Audit Logs
```

Never expose documents without permission.

---

# Observability

Track

```text
Retrieval Accuracy

Embedding Latency

Vector Search Latency

LLM Latency

Token Usage

Cost

Cache Hit Rate
```

---

# Scaling The System

Scale

```text
Retriever

LLM

Vector DB

Separately
```

Use

```text
Load Balancers

Autoscaling

Caching
```

---

# Common Production Problems

Poor Chunking

↓

Poor Retrieval

---

Wrong Embedding Model

↓

Low Recall

---

No Metadata

↓

Security Issues

---

No Reranker

↓

Irrelevant Answers

---

Prompt Too Large

↓

Context Overflow

---

No Cache

↓

High Cost

---

# End-to-End Architecture

```text
                    User
                      │
                      ▼
                API Gateway
                      │
                      ▼
              Authentication
                      │
                      ▼
                 RAG Service
                      │
        ┌─────────────┼─────────────┐
        ▼             ▼             ▼
 Conversation     Query         Cache
    Memory      Embedding
                      │
                      ▼
              Hybrid Retriever
          ┌───────────┴───────────┐
          ▼                       ▼
     Vector Search          BM25 Search
          │                       │
          └───────────┬───────────┘
                      ▼
                 Reranker
                      ▼
               Prompt Builder
                      ▼
                    LLM
                      ▼
              Output Guardrails
                      ▼
                Audit Logging
                      ▼
                   Response
```

---

# Enterprise Case Study

Employee asks

```text
How many maternity leave days are available?
```

Execution

```text
Authenticate User

↓

Embed Query

↓

Hybrid Search

↓

Retrieve Top 20

↓

Rerank Top 5

↓

Prompt Builder

↓

LLM

↓

Generate Citation

↓

Audit Log

↓

Return Answer
```

---

# Common Interview Questions

---

## Question

Why RAG instead of Fine-Tuning?

### Answer

```text
RAG separates knowledge from the model, making updates faster, cheaper, and easier.
```

---

## Question

Why Chunk Documents?

### Answer

```text
Large documents exceed embedding and context limits while reducing retrieval precision.
```

---

## Question

Why Hybrid Search?

### Answer

```text
It combines semantic understanding with exact keyword matching, improving retrieval quality.
```

---

## Question

Why Rerank Results?

### Answer

```text
The retriever maximizes recall, while the reranker improves precision before passing context to the LLM.
```

---

## Question

Why Store Metadata?

### Answer

```text
Metadata enables filtering, access control, versioning, and better retrieval.
```

---

## Question

Why Cache Embeddings?

### Answer

```text
Embedding generation is computationally expensive, so caching reduces latency and cost.
```

---

# Chapter 2 Summary

## Core Concepts Covered

```text
Enterprise RAG

Document Ingestion

OCR

Parsing

Cleaning

Metadata Extraction

Chunking

Embedding Pipeline

Vector Database

ANN Indexes

Hybrid Search

Metadata Filtering

Top-K Retrieval

Reranking

Prompt Builder

Conversation Memory

Citation Generation

Caching

Security

Observability

Scaling

Enterprise Architecture
```

---

## Memory Map

```text
Documents
      ↓

Ingestion
      ↓

Chunking
      ↓

Embeddings
      ↓

Vector DB
      ↓

Hybrid Retrieval
      ↓

Reranker
      ↓

Prompt Builder
      ↓

LLM
      ↓

Guardrails
      ↓

Response
```

---

## Interview Cheat Sheet

```text
RAG
↓
Retrieve Before Generate

Chunking
↓
Split Documents

Embeddings
↓
Vectors

Vector DB
↓
Semantic Search

Hybrid Search
↓
Vector + BM25

Reranker
↓
Improve Precision

Metadata
↓
Filtering & Security

Prompt Builder
↓
Context Assembly

Citations
↓
Grounded Responses

Enterprise RAG
↓
Secure + Scalable + Observable
```

---

## Next Chapter

```text
Chapter 3

Agentic AI System Design

Planner

Supervisor

Memory

Tool Layer

Multi-Agent

State Management

Human Approval

Observability

Production Architecture
```

In the next chapter, we'll design a **production-grade Agentic AI platform**, integrating everything from Module 8—planning, memory, multi-agent orchestration, and secure tool execution—into a complete enterprise architecture.